# Checking for suitable data
This notebook is here to check whether suitable data is available for urban vegetation change in Singapore. 
- RQ: Where has the urban vegetation in-/decreased between year X and year Y, plus do the gains offset the losses?

1. First checking with openEO backend the CDSE data catalogue

In [1]:
import openeo
import rasterio
from localtileserver import TileClient, get_leaflet_tile_layer
import leafmap

In [2]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()

Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

1.1. Checking Sentinel2 Data

In [3]:
connection.describe_collection("SENTINEL2_L2A")

{'bands': [{'name': 'B01'},
  {'name': 'B02'},
  {'name': 'B03'},
  {'name': 'B04'},
  {'name': 'B05'},
  {'name': 'B06'},
  {'name': 'B07'},
  {'name': 'B08'},
  {'name': 'B8A'},
  {'name': 'B09'},
  {'name': 'B11'},
  {'name': 'B12'},
  {'name': 'WVP'},
  {'name': 'AOT'},
  {'name': 'SCL'},
  {'name': 'sunAzimuthAngles'},
  {'name': 'sunZenithAngles'},
  {'name': 'viewAzimuthMean'},
  {'name': 'viewZenithMean'},
  {'name': 'CLD'},
  {'name': 'SNW'}],
 'ceosard:specification': 'SR',
 'ceosard:specification_version': '5.0.1',
 'ceosard:type': 'optical',
 'cube:dimensions': {'bands': {'type': 'bands',
   'values': ['B01',
    'B02',
    'B03',
    'B04',
    'B05',
    'B06',
    'B07',
    'B08',
    'B8A',
    'B09',
    'B11',
    'B12',
    'WVP',
    'AOT',
    'SCL',
    'sunAzimuthAngles',
    'sunZenithAngles',
    'viewAzimuthMean',
    'viewZenithMean',
    'CLD',
    'SNW']},
  't': {'extent': ['2015-06-27T10:25:31Z', None], 'type': 'temporal'},
  'x': {'axis': 'x',
   'extent': [-180, 180],
   'reference_system': {'$schema': 'https://proj.org/schemas/v0.2/projjson.schema.json',
    'area': 'World',
    'bbox': {'east_longitude': 180,
     'north_latitude': 90,
     'south_latitude': -90,
     'west_longitude': -180},
    'coordinate_system': {'axis': [{'abbreviation': 'Lat',
       'direction': 'north',
       'name': 'Geodetic latitude',
       'unit': 'degree'},
      {'abbreviation': 'Lon',
       'direction': 'east',
       'name': 'Geodetic longitude',
       'unit': 'degree'}],
     'subtype': 'ellipsoidal'},
    'datum': {'ellipsoid': {'inverse_flattening': 298.257223563,
      'name': 'WGS 84',
      'semi_major_axis': 6378137},
     'name': 'World Geodetic System 1984',
     'type': 'GeodeticReferenceFrame'},
    'id': {'authority': 'OGC', 'code': 'Auto42001', 'version': '1.3'},
    'name': 'AUTO 42001 (Universal Transverse Mercator)',
    'type': 'GeodeticCRS'},
   'type': 'spatial'},
  'y': {'axis': 'y',
   'extent': [-90, 90],
   'reference_system': {'$schema': 'https://proj.org/schemas/v0.2/projjson.schema.json',
    'area': 'World',
    'bbox': {'east_longitude': 180,
     'north_latitude': 90,
     'south_latitude': -90,
     'west_longitude': -180},
    'coordinate_system': {'axis': [{'abbreviation': 'Lat',
       'direction': 'north',
       'name': 'Geodetic latitude',
       'unit': 'degree'},
      {'abbreviation': 'Lon',
       'direction': 'east',
       'name': 'Geodetic longitude',
       'unit': 'degree'}],
     'subtype': 'ellipsoidal'},
    'datum': {'ellipsoid': {'inverse_flattening': 298.257223563,
      'name': 'WGS 84',
      'semi_major_axis': 6378137},
     'name': 'World Geodetic System 1984',
     'type': 'GeodeticReferenceFrame'},
    'id': {'authority': 'OGC', 'code': 'Auto42001', 'version': '1.3'},
    'name': 'AUTO 42001 (Universal Transverse Mercator)',
    'type': 'GeodeticCRS'},
   'type': 'spatial'}},
 'description': 'SENTINEL-2 is a wide-swath, high-resolution, multi-spectral imaging mission, supporting Copernicus Land Monitoring studies, including the monitoring of vegetation, soil and water cover, as well as observation of inland waterways and coastal areas. The SENTINEL-2 Multispectral Instrument (MSI) samples 13 spectral bands: four bands at 10 metres, six bands at 20 metres and three bands at 60 metres spatial resolution.\n The level 2A data is atmospherically corrected using Sen2Cor.',
 'extent': {'spatial': {'bbox': [[-180, -90, 180, 90]]},
  'temporal': {'interval': [['2015-06-27T10:25:31Z', None]]}},
 'id': 'SENTINEL2_L2A',
 'keywords': ['COPERNICUS',
  'ESA',
  'Orthoimagery',
  'Sentinel-2',
  'MSI',
  'Level-2A',
  'Radiometry',
  'Plant Resource',
  'TOC'],
 'license': 'other',
 'links': [{'href': 'https://sentinels.copernicus.eu/documents/247904/690755/Sentinel_Data_Legal_Notice',
   'rel': 'license',
   'title': 'Legal notice on the use of Copernicus Sentinel Data and Service Information',
   'type': 'application/pdf'},
  {'href': 'https://stac.dataspace.cop

In [4]:
sentinel2_L2A_cube = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent={"west": 103.60, "south": 1.20, "east": 104.05, "north": 1.48},
    temporal_extent=["2016-01-01", "2016-12-31"],
    bands=["B04", "B08", "SCL"], # red, nir, scene classification
    max_cloud_cover=70,)

In [ ]:
# Mask out cloud shadows, clouds and cirrus using the scene classification band
# SCL classification fromhttps://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/scene-classification/
scl = sentinel2_L2A_cube.band("SCL")
# 3 = cloud shadows, 8 = cloud medium probability, 9 = cloud high probability, 10 = thin cirrus
removed_scl_classes = (scl == 3) | (scl == 8) | (scl == 9) | (scl == 10)

sentinel2_L2A_cube = sentinel2_L2A_cube.mask(removed_scl_classes)

In [6]:
# median (due to potential outliers) composite over the year
sentinel2_L2A_composite = sentinel2_L2A_cube.reduce_dimension(dimension="t", reducer="median")

# Calculate NDVI
red = sentinel2_L2A_composite.band("B04")
nir = sentinel2_L2A_composite.band("B08")

ndvi = (nir - red) / (nir + red)

ndvi.download("../data/raw/raster/singapore_ndvi_s2_2016.tif")

KeyboardInterrupt: 

In [7]:
map = leafmap.Map(center=[1.35, 103.82], zoom=11)
map.add_raster("C:\\Users\\clair\\OneDrive\\Dokumente\\University Zurich\\Semester 7 (HS)\\sds320\\project\\data\\raw\\raster\\singapore_ndvi_s2_2016.tif", colormap="RdYlGn", vmin=-0.2, vmax=0.9)
map

Map(center=[1.340029, 103.8249635], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…